In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold

from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,classification_report)

from imblearn.over_sampling import SMOTE

from lightgbm import LGBMClassifier

In [2]:
df = pd.read_csv(r"E:\AARAV\Infotact-DS-ML\Project-1-Predictive-Maintenance\data\processed\model_ready_dataset.csv")

df.head()

,HDF,OSF,PWF,TWF,rpm_torque_interaction,Rotational speed [rpm],load_stress,Torque [Nm],load_density,temperature_ratio,Tool wear [min],tool_wear_mean_10,temperature_difference,air_temp_mean_10,UDI,Machine failure
0,0,0,0,0,71177.0,1306,29.7025,54.5,0.545,1.034806,50,36.8,10.4,298.60,19,0
1,0,0,0,0,53040.0,1632,10.5625,32.5,0.325,1.034794,55,40.2,10.4,298.64,20,0
2,0,0,0,0,58712.5,1375,18.2329,42.7,0.427,1.034794,58,43.6,10.4,298.69,21,0
3,0,0,0,0,64960.0,1450,20.0704,44.8,0.448,1.035141,63,47.0,10.5,298.71,22,0
4,0,0,0,0,48536.7,1581,9.4249,30.7,0.307,1.034794,65,50.1,10.4,298.74,23,0


In [3]:
df.shape

(9982, 16)

In [4]:
df["Machine failure"].value_counts()

Machine failure
0    9643
1     339
Name: count, dtype: int64

In [5]:
failure_percentage = (df["Machine failure"].value_counts(normalize=True)*100)

failure_percentage

Machine failure
0    96.603887
1     3.396113
Name: proportion, dtype: float64

In [6]:
df.columns = (
    df.columns
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("{", "", regex=False)
    .str.replace("}", "", regex=False)
    .str.replace(":", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace('"', "", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace("/", "_", regex=False)
)

X = df.drop("Machine failure",axis=1)

y = df["Machine failure"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

In [8]:
fold_results = []


for fold, (train_index, test_index) in enumerate(skf.split(X, y)):

    print(f"Training Fold {fold+1}")


    # Split fold data
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]

    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]


    # Apply SMOTE only on training data
    smote = SMOTE(random_state=42)

    X_train_resampled, y_train_resampled = smote.fit_resample(X_train,y_train)


    # LightGBM model
    model = LGBMClassifier(random_state=42)


    model.fit(X_train_resampled,y_train_resampled)


    # Prediction
    y_pred = model.predict(X_test)

    # Metrics
    fold_results.append({
                        "Fold": fold+1,
                        "Accuracy": accuracy_score(y_test,y_pred),
                        "Precision": precision_score(y_test,y_pred),
                        "Recall": recall_score(y_test,y_pred),
                        "F1": f1_score(y_test,y_pred)
                    })

Training Fold 1
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 7714, number of negative: 7714
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002350 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2805
[LightGBM] [Info] Number of data points in the train set: 15428, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training Fold 2
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 7714, number of negative: 7714
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001887 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2804
[LightGBM] [Info] Number of data points in

In [9]:
results_df = pd.DataFrame(fold_results)

In [10]:
results_df

,Fold,Accuracy,Precision,Recall,F1
0,1,0.995493,0.904110,0.970588,0.936170
1,2,0.992489,0.827160,0.985294,0.899329
2,3,0.993487,0.855263,0.970149,0.909091
3,4,0.995992,0.916667,0.970588,0.942857
4,5,0.990481,0.802469,0.955882,0.872483


In [13]:
mean_scores = results_df.mean()

mean_scores

Fold         3.000000
Accuracy     0.993588
Precision    0.861134
Recall       0.970500
F1           0.911986
dtype: float64

In [14]:
results_df.to_csv(r"E:\AARAV\Infotact-DS-ML\Project-1-Predictive-Maintenance\data\processed\week3_cv_results.csv",index=False)

In [15]:
import joblib

In [16]:
final_model = LGBMClassifier(random_state=42)

smote = SMOTE(random_state=42)

X_balanced, y_balanced = smote.fit_resample(X, y)

final_model.fit(X_balanced,y_balanced)



[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 9643, number of negative: 9643
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003387 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2807
[LightGBM] [Info] Number of data points in the train set: 19286, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


LGBMClassifier(random_state=42)

In [17]:
joblib.dump(final_model, r"E:\AARAV\Infotact-DS-ML\Project-1-Predictive-Maintenance\models\lightgbm_model.pkl")

['E:\\AARAV\\Infotact-DS-ML\\Project-1-Predictive-Maintenance\\models\\lightgbm_model.pkl']